# Coupled Heart–Brain Hopf Pipeline (Refactored)

Self-contained notebook: correct oscillator equations, differentiable feedback, no reset_weights.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, detrend
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import mne
from torchdiffeq import odeint_adjoint

proj_root = Path.cwd()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# --- Config ---
CFG = {
    'data': {'ecg_path': 'transdef_mf2pt2_rest_raw.fif', 'eeg_path': 'scout_id_309.mat', 'sc_path': 'SC_CC120309-27.mat',
             'ecg_channel': 322, 't_start': 2000, 't_end': 4000, 'fs_raw': 1000},
    'preprocessing': {'ecg_lowcut': 1.5, 'ecg_highcut': 20, 'eeg_lowcut': 0.5, 'eeg_highcut': 20, 'ecg_negate': True},
    'dynamics': {'heart': {'omega1_hz': 1.0, 'omega2_hz': 1.2}, 'brain': {'osc_per_region': 3, 'eta_omega': 0.05, 'eta_alpha': 0.005, 'eta_theta': 0.05}},
    'training': {'heart_epochs': 25000, 'brain_epochs': 30, 'mlp_epochs': 100, 'feedback_epochs': 5000, 'seed': 42,
                 'heart_lr': 1e-3, 'mlp_lr': 1e-2, 'feedback_lr': 1e-3, 'log_interval': 500},
    'model': {'ecg_to_brain': {'ecg_dim': 50, 'n_vns': 64, 'hidden_dim': 64}, 'oscillator': {'T': 2.0, 'fs': 100}},
    'target_indices': [4],
}

In [ ]:
# --- Utils ---
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_random_frequencies(num_regions, osc_per_region, low_hz=1, high_hz=20, seed=None):
    if seed is not None:
        np.random.seed(seed)
    total = num_regions * osc_per_region
    return 2 * np.pi * np.random.uniform(low_hz, high_hz, total)

def expand_structural_connectivity(Sc_region, osc_per_region, intra_value=0.0001, seed=None):
    if seed is not None:
        np.random.seed(seed)
    N = Sc_region.shape[0] * osc_per_region
    Sc_full = np.zeros((N, N))
    for i in range(Sc_region.shape[0]):
        for j in range(Sc_region.shape[0]):
            si, ei = i * osc_per_region, (i + 1) * osc_per_region
            sj, ej = j * osc_per_region, (j + 1) * osc_per_region
            if i == j:
                Sc_full[si:ei, sj:ej] = intra_value
            else:
                rb = np.random.rand(osc_per_region, osc_per_region)
                Sc_full[si:ei, sj:ej] = rb * Sc_region[i, j] / (rb.sum() + 1e-9)
    np.fill_diagonal(Sc_full, 0.0)
    return Sc_full

In [ ]:
# --- Data loading ---
def preprocess_signal(signal, fs=1000, lowcut=1.5, highcut=20):
    detrended = detrend(signal)
    b, a = butter(4, [lowcut / (0.5 * fs), highcut / (0.5 * fs)], btype='band')
    filtered = filtfilt(b, a, detrended)
    return (filtered - np.mean(filtered)) / (np.std(filtered) + 1e-12)

def load_all_data(base_dir):
    base = Path(base_dir)
    raw = mne.io.read_raw_fif(str(base / CFG['data']['ecg_path']), preload=False)
    data, _ = raw[CFG['data']['ecg_channel'], CFG['data']['t_start']:CFG['data']['t_end']]
    ecg = -data[0] if CFG['preprocessing']['ecg_negate'] else data[0]
    eeg = loadmat(str(base / CFG['data']['eeg_path']))['Value'][:, CFG['data']['t_start']:CFG['data']['t_end']]
    sc_data = loadmat(str(base / CFG['data']['sc_path']))
    sc = sc_data['sc'].astype(np.float64)
    max_val = np.max(sc)
    Sw = (sc / max_val) * 0.01 if max_val > 0 else sc
    non_zero = [np.nonzero(Sw[i, :])[0] for i in range(Sw.shape[0])]
    return ecg.astype(np.float64), eeg.astype(np.float64), Sw, non_zero

ecg_raw, eeg_raw, sc_matrix, non_zero = load_all_data(proj_root)
ecg_processed = preprocess_signal(ecg_raw, CFG['data']['fs_raw'], CFG['preprocessing']['ecg_lowcut'], CFG['preprocessing']['ecg_highcut'])
eeg_processed = np.array([preprocess_signal(row, CFG['data']['fs_raw'], CFG['preprocessing']['eeg_lowcut'], CFG['preprocessing']['eeg_highcut']) for row in eeg_raw])
print(f'ECG: {ecg_processed.shape}, EEG: {eeg_processed.shape}')

In [ ]:
# --- Dynamics: Heart oscillators (NumPy for pre-train) ---
def simulate_coupled_oscillators_numpy(T=2, dt=0.01, omega1_hz=1.0, omega2_hz=1.2, alpha=1, A_init=0.0001, theta_init=3.14, n=1, modulation=None):
    omega1, omega2 = 2 * np.pi * omega1_hz, 2 * np.pi * omega2_hz
    N = int(T / dt)
    r1, r2, phi1, phi2 = 1.0, 1.0, 0.0, 0.0
    A12, A21 = A_init, A_init
    R1, R2, Phi1, Phi2 = np.zeros(N), np.zeros(N), np.zeros(N), np.zeros(N)
    for i in range(N):
        R1[i], R2[i], Phi1[i], Phi2[i] = r1, r2, phi1, phi2
        p12, p21 = theta_init + n * (phi2 - phi1), theta_init + n * (phi1 - phi2)
        dr1 = alpha * r1 - r1**3 + A12 * r2 * np.cos(p12)
        dr2 = alpha * r2 - r2**3 + A21 * r1 * np.cos(p21)
        if modulation is not None and i < len(modulation):
            dr1 += 0.1 * modulation[i, 0]
            dr2 += 0.1 * modulation[i, 1]
        dphi1 = omega1 + A12 * r2 / (r1 + 1e-8) * np.sin(p12)
        dphi2 = omega2 + A21 * r1 / (r2 + 1e-8) * np.sin(p21)
        r1, r2 = np.clip(r1 + dr1 * dt, 0.01, 2.0), np.clip(r2 + dr2 * dt, 0.01, 2.0)
        phi1, phi2 = phi1 + dphi1 * dt, phi2 + dphi2 * dt
    return np.stack((R1*np.cos(Phi1), R1*np.sin(Phi1), R2*np.cos(Phi2), R2*np.sin(Phi2)), axis=1)

In [ ]:
# --- Dynamics: Heart oscillators (PyTorch, differentiable) ---
class HeartOscillatorTorch(nn.Module):
    def __init__(self, omega1_hz=1.0, omega2_hz=1.2, alpha=1, A_init=0.0001, theta_init=3.14, n=1):
        super().__init__()
        self.omega1 = 2 * np.pi * omega1_hz
        self.omega2 = 2 * np.pi * omega2_hz
        self.alpha, self.A12, self.A21 = alpha, A_init, A_init
        self.theta12, self.theta21, self.n = theta_init, theta_init, n

    def forward(self, T, dt, modulation):
        squeeze = modulation.dim() == 2
        if squeeze:
            modulation = modulation.unsqueeze(0)
        batch, n_steps = modulation.shape[0], modulation.shape[1]
        r1 = torch.ones(batch, 1, device=modulation.device, dtype=modulation.dtype)
        r2 = torch.ones(batch, 1, device=modulation.device, dtype=modulation.dtype)
        phi1 = torch.zeros(batch, 1, device=modulation.device, dtype=modulation.dtype)
        phi2 = torch.zeros(batch, 1, device=modulation.device, dtype=modulation.dtype)
        outputs = []
        for i in range(n_steps):
            mod = modulation[:, i, :]
            p12 = self.theta12 + self.n * (phi2 - phi1)
            p21 = self.theta21 + self.n * (phi1 - phi2)
            dr1 = (self.alpha - r1**2) * r1 + self.A12 * r2 * torch.cos(p12) + 0.1 * mod[:, 0:1]
            dr2 = (self.alpha - r2**2) * r2 + self.A21 * r1 * torch.cos(p21) + 0.1 * mod[:, 1:2]
            dphi1 = self.omega1 + self.A12 * (r2 / (r1 + 1e-8)) * torch.sin(p12)
            dphi2 = self.omega2 + self.A21 * (r1 / (r2 + 1e-8)) * torch.sin(p21)
            r1 = torch.clamp(r1 + dr1 * dt, 0.01, 2.0)
            r2 = torch.clamp(r2 + dr2 * dt, 0.01, 2.0)
            phi1, phi2 = phi1 + dphi1 * dt, phi2 + dphi2 * dt
            outputs.append(torch.cat([r1*torch.cos(phi1), r1*torch.sin(phi1), r2*torch.cos(phi2), r2*torch.sin(phi2)], dim=-1))
        out = torch.stack(outputs, dim=1)
        return out.squeeze(0) if squeeze else out

In [ ]:
# --- Models ---
class HeartModel(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=100, feature_dim=50, output_dim=1):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Sigmoid(),
            nn.Linear(hidden_dim, hidden_dim), nn.Sigmoid(),
            nn.Linear(hidden_dim, feature_dim))
        self.output_layer = nn.Linear(feature_dim, output_dim)
    def forward(self, x):
        return self.output_layer(self.feature_extractor(x))
    def get_features(self, x):
        return self.feature_extractor(x)

class FeedbackMLP(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, output_dim))
    def forward(self, rcos_phi):
        if rcos_phi.dim() == 1:
            rcos_phi = rcos_phi.unsqueeze(-1)
        return 0.5 * torch.tanh(self.net(rcos_phi))

In [ ]:
# --- OscillatorLayer: correct Hopf equations ---
# dr_i/dt = (mu - r_i^2)*r_i + k*sum_j C_ij*r_j*cos(phi_j - phi_i - theta_ij) + ECG_amp_i
# dphi_i/dt = omega_i + k*sum_j C_ij*(r_j/r_i)*sin(phi_j - phi_i - theta_ij) + ECG_phase_i
class OscillatorLayer(nn.Module):
    def __init__(self, N_osc=64, T=2, fs=100, coupling_sparsity=0.3, coupling_strength=0.05, freq_hz_min=2, freq_hz_max=10, seed=42, dev='cpu'):
        super().__init__()
        self.N_osc = N_osc
        self.num_steps = int(T * fs)
        self.dt = 1.0 / fs
        self.mu = nn.Parameter(torch.tensor(1.0))
        torch.manual_seed(seed)
        self.omega = nn.Parameter(2 * np.pi * (freq_hz_min + torch.rand(N_osc) * (freq_hz_max - freq_hz_min)))
        self.initial_r = nn.Parameter(torch.ones(N_osc) * 0.1)
        self.initial_phi = nn.Parameter(torch.zeros(N_osc))
        torch.manual_seed(seed + 1)
        mask = torch.rand(N_osc, N_osc) > coupling_sparsity
        mask.fill_diagonal_(False)
        self.register_buffer('C', torch.rand(N_osc, N_osc) * 0.02 * mask.float())
        self.register_buffer('k', torch.tensor(coupling_strength))
        tr = torch.rand(N_osc, N_osc) * 2 * np.pi - np.pi
        self.theta = nn.Parameter(tr - tr.T)

    def forward(self, ecg_amp, ecg_phase=None):
        batch = ecg_amp.shape[0]
        r = self.initial_r.unsqueeze(0).repeat(batch, 1).unsqueeze(-1)
        phi = self.initial_phi.unsqueeze(0).repeat(batch, 1).unsqueeze(-1)
        ecg_phase = ecg_phase if ecg_phase is not None else torch.zeros_like(ecg_amp)
        ecg_phase = ecg_phase.unsqueeze(-1)
        for _ in range(self.num_steps):
            phi_diff = phi.transpose(-2, -1) - phi - self.theta.unsqueeze(0)
            r_j = r.transpose(-2, -1).expand(-1, self.N_osc, -1)
            r_i_safe = torch.clamp(r, 1e-6, 10.0).expand(-1, -1, self.N_osc)
            coupling_r = self.k * torch.sum(self.C.unsqueeze(0) * r_j * torch.cos(phi_diff), dim=-1).unsqueeze(-1)
            coupling_phi = self.k * torch.sum(self.C.unsqueeze(0) * (r_j / r_i_safe) * torch.sin(phi_diff), dim=-1).unsqueeze(-1)
            dr = (self.mu - r**2) * r + coupling_r + ecg_amp.unsqueeze(-1)
            dphi = self.omega.unsqueeze(0).unsqueeze(-1) + coupling_phi + ecg_phase
            r = torch.clamp(r + dr * self.dt, 1e-6, 10.0)
            phi = phi + dphi * self.dt
        return torch.cat([r.squeeze(-1)*torch.cos(phi.squeeze(-1)), r.squeeze(-1)*torch.sin(phi.squeeze(-1))], dim=-1)

In [ ]:
# OscillatorLayer defined above with correct Hopf dynamics

In [ ]:
# --- ECGToOscillatorMLP ---
class ECGToOscillatorMLP(nn.Module):
    def __init__(self, ecg_dim=50, N_VNS=64, hidden_dim=64, output_dim=16, T=2, fs=100, dev='cpu'):
        super().__init__()
        self.pre_osc = nn.Sequential(
            nn.Linear(ecg_dim, hidden_dim), nn.Sigmoid(),
            nn.Linear(hidden_dim, hidden_dim), nn.Sigmoid(),
            nn.Linear(hidden_dim, N_VNS * 2))
        self.osc_layer = OscillatorLayer(N_osc=N_VNS, T=T, fs=fs, dev=dev)
        self.post_osc = nn.Sequential(
            nn.Linear(N_VNS * 2, hidden_dim), nn.Sigmoid(),
            nn.Linear(hidden_dim, hidden_dim), nn.Sigmoid(),
            nn.Linear(hidden_dim, output_dim))

    def forward(self, ecg_features):
        if ecg_features.dim() == 1:
            ecg_features = ecg_features.unsqueeze(0)
        pre = self.pre_osc(ecg_features)
        amp, phase = pre[:, :pre.shape[-1]//2], pre[:, pre.shape[-1]//2:]
        osc_out = self.osc_layer(amp, phase)
        out = self.post_osc(osc_out)
        return out.squeeze(0) if out.shape[0] == 1 else out

In [ ]:
# --- Brain ODE (phase_diff = phi_j - phi_i - theta_ij) ---
class ODEFunc(nn.Module):
    def __init__(self, mu, eta_theta, eta_omega, eta_alpha, N, Sc, D_tensor, t_eval, mlp_model=None, hidden_repr=None):
        super().__init__()
        self.mu, self.eta_theta, self.eta_omega, self.eta_alpha, self.N = mu, eta_theta, eta_omega, eta_alpha, N
        self.register_buffer('Sc', Sc if isinstance(Sc, torch.Tensor) else torch.tensor(Sc, dtype=torch.float32))
        self.register_buffer('D_tensor', D_tensor)
        self.register_buffer('t_eval', t_eval)
        self.mlp_model, self.hidden_repr = mlp_model, hidden_repr

    def forward(self, t, state):
        N = self.N
        r, phi = state[:N], state[N:2*N]
        theta = state[2*N:2*N+N**2].view(N, N)
        omega, alpha = state[2*N+N**2:3*N+N**2], state[3*N+N**2:4*N+N**2]
        omega_safe = torch.clamp(omega, 2*np.pi*0.5, 2*np.pi*20)
        r, alpha = torch.clamp(r, 1e-1, 2.0), torch.clamp(alpha, -1.0, 1.0)
        r_safe = torch.clamp(r, 1e-5, 10.0)
        phase_diff = phi[None, :] - phi[:, None] - theta
        t_idx = (self.t_eval - t.item()).abs().argmin().item()
        D = self.D_tensor[t_idx]
        P = torch.sum(alpha * r * torch.cos(phi))
        e = D - P
        ecg_input = torch.zeros(N, device=state.device, dtype=state.dtype)
        if self.mlp_model is not None and self.hidden_repr is not None:
            idx = max(0, min(int((t/self.t_eval[-1]).item() * (self.hidden_repr.shape[0]-1)), self.hidden_repr.shape[0]-1))
            feats = self.hidden_repr[idx].to(state.device)
            ecg_input = torch.clamp(self.mlp_model(feats).squeeze(), 0.01, 5.0)
        coupling_r = torch.sum(torch.abs(self.Sc) * r[None, :] * torch.cos(phase_diff), dim=1)
        coupling_phi = torch.sum(torch.abs(self.Sc) * (r[None, :] / (r_safe[:, None] + 1e-8)) * torch.sin(phase_diff), dim=1)
        drdt = (self.mu - r**2)*r + coupling_r + e*torch.cos(phi) + ecg_input
        dphidt = omega_safe + coupling_phi - (e / (r_safe + 1e-8)) * torch.sin(phi)
        dthetadt = self.eta_theta * torch.sin(phase_diff) * torch.abs(self.Sc)
        domegadt = -self.eta_omega * e * torch.sin(phi)
        dalphadt = self.eta_alpha * e * r * torch.cos(phi)
        for x in [drdt, dphidt, dthetadt, domegadt, dalphadt]:
            torch.clamp_(x, -1e2, 1e2)
        return torch.cat([drdt, dphidt, dthetadt.flatten(), domegadt, dalphadt])

class TorchRevHopfNetwork:
    def __init__(self, mu, eta_omega, eta_alpha, eta_theta, D_tensor, t_eval, N, Sc, mlp_model=None, hidden_repr=None, device=None):
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.N = N
        Sc = Sc if isinstance(Sc, torch.Tensor) else torch.tensor(Sc, device=self.device, dtype=torch.float32)
        D_tensor = D_tensor.to(self.device)
        t_eval = t_eval.to(self.device)
        hr = hidden_repr.to(self.device) if hidden_repr is not None else None
        self.ode_func = ODEFunc(mu, eta_theta, eta_omega, eta_alpha, N, Sc, D_tensor, t_eval, mlp_model, hr).to(self.device)

    def solve(self, r0, phi0, theta0, omega0, alpha0):
        y0 = torch.tensor(np.concatenate([r0, phi0, theta0.flatten(), omega0, alpha0]), device=self.device, dtype=torch.float32)
        sol = odeint_adjoint(self.ode_func, y0, self.ode_func.t_eval, method='rk4', rtol=1e-5, atol=1e-7)
        N = self.N
        r, phi = sol[:, :N], sol[:, N:2*N]
        theta = sol[:, 2*N:2*N+N**2].view(-1, N, N)
        omega = sol[:, 2*N+N**2:3*N+N**2]
        alpha = sol[:, 3*N+N**2:4*N+N**2]
        rcos_phi = torch.sum(r * torch.cos(phi), dim=1)
        return r, phi, theta, omega, alpha, rcos_phi

In [ ]:
# --- Training ---
set_seed(CFG['training']['seed'])
dyn = CFG['dynamics']['heart']
tr = CFG['training']

print('Stage 0: Heart pre-training')
heart_model = HeartModel().to(device)
opt = optim.Adam(heart_model.parameters(), lr=tr['heart_lr'])
sim_osc = simulate_coupled_oscillators_numpy(T=2, dt=0.01, omega1_hz=dyn['omega1_hz'], omega2_hz=dyn['omega2_hz'])
sim_t = torch.tensor(sim_osc, dtype=torch.float32, device=device)
ecg_tgt = torch.tensor(ecg_processed[::10], dtype=torch.float32, device=device).unsqueeze(1)
for ep in range(tr['heart_epochs']):
    loss = nn.MSELoss()(heart_model(sim_t), ecg_tgt)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (ep+1) % 2500 == 0:
        print(f'  Epoch {ep+1}, Loss: {loss.item():.6f}')
print('Heart done.')

In [ ]:
# ECG features for brain
with torch.no_grad():
    sim_osc = simulate_coupled_oscillators_numpy(T=2, dt=0.01, omega1_hz=dyn['omega1_hz'], omega2_hz=dyn['omega2_hz'])
    hidden_repr = heart_model.get_features(torch.tensor(sim_osc, dtype=torch.float32, device=device))
print(f'hidden_repr: {hidden_repr.shape}')

In [ ]:
# Stage 1: Brain pre-training
target_idx = CFG['target_indices'][0]
t_duration, fs = 2.0, 100
t = np.arange(0, t_duration, 1/fs)
target_signal = eeg_processed[target_idx, ::10]
connected = np.unique(np.append(non_zero[target_idx], target_idx))
osc_per_region = CFG['dynamics']['brain']['osc_per_region']
N = len(connected) * osc_per_region
Sc_reduced = expand_structural_connectivity(sc_matrix[np.ix_(connected, connected)], osc_per_region, seed=tr['seed'])
omega_full = get_random_frequencies(68, osc_per_region, 1, 20, tr['seed'])
alpha_full = np.random.uniform(0.1, 0.7, 68 * osc_per_region)
omega0 = np.concatenate([omega_full[i*osc_per_region:(i+1)*osc_per_region] for i in connected])
alpha0 = np.clip(np.concatenate([alpha_full[i*osc_per_region:(i+1)*osc_per_region] for i in connected]), 0.05, 0.5)
r0,phi0 = 0.1*np.ones(N), np.zeros(N)
theta0 = np.pi*(2*np.random.rand(N,N)-1) - np.pi*(2*np.random.rand(N,N)-1).T

D_tensor = torch.tensor(target_signal, dtype=torch.float32, device=device)
t_eval = torch.tensor(t, dtype=torch.float32, device=device)
model = TorchRevHopfNetwork(mu=1, eta_omega=0.05, eta_alpha=0.005, eta_theta=0.05, D_tensor=D_tensor, t_eval=t_eval, N=N, Sc=Sc_reduced, device=device)
brain_losses = []
for ep in range(tr['brain_epochs']):
    with torch.no_grad():
        r, phi, theta, omega, alpha, _ = model.solve(r0, phi0, theta0, omega0, alpha0)
        loss = nn.MSELoss()(torch.sum(alpha*r*torch.cos(phi), dim=1), D_tensor)
        brain_losses.append(loss.item())
        theta0, omega0, alpha0 = theta[-1].cpu().numpy(), omega[-1].cpu().numpy(), alpha[-1].cpu().numpy()
    if (ep+1) % 10 == 0:
        print(f'Brain Epoch {ep+1}, Loss: {loss.item():.6f}')
brain_params = {'r': r0, 'phi': phi0, 'theta': theta0, 'omega': omega0, 'alpha': alpha0}
print('Brain done.')

In [ ]:
# Stage 2: ECG -> Oscillator -> Brain
ecg_to_osc_mlp = ECGToOscillatorMLP(ecg_dim=50, N_VNS=64, hidden_dim=64, output_dim=N, T=2, fs=100, dev=device).to(device)
opt = optim.Adam(ecg_to_osc_mlp.parameters(), lr=tr['mlp_lr'])
model = TorchRevHopfNetwork(mu=1, eta_omega=0, eta_alpha=0, eta_theta=0, D_tensor=D_tensor, t_eval=t_eval, N=N, Sc=Sc_reduced,
    mlp_model=ecg_to_osc_mlp, hidden_repr=hidden_repr, device=device)
mlp_losses = []
for ep in range(tr['mlp_epochs']):
    r, phi, theta, omega, alpha, _ = model.solve(brain_params['r'], brain_params['phi'], brain_params['theta'], brain_params['omega'], brain_params['alpha'])
    loss = nn.MSELoss()(torch.sum(alpha*r*torch.cos(phi), dim=1), D_tensor)
    opt.zero_grad()
    loss.backward()
    opt.step()
    mlp_losses.append(loss.item())
    if (ep+1) % 20 == 0:
        print(f'MLP Epoch {ep+1}, Loss: {loss.item():.6f}')
print('MLP done.')

In [ ]:
# Extract rcos_phi for feedback
model_final = TorchRevHopfNetwork(mu=1, eta_omega=0, eta_alpha=0, eta_theta=0, D_tensor=D_tensor, t_eval=t_eval, N=N, Sc=Sc_reduced,
    mlp_model=ecg_to_osc_mlp, hidden_repr=hidden_repr, device=device)
r_final, phi_final, theta_final, omega_final, alpha_final, rcos_phi_final = model_final.solve(
    brain_params['r'], brain_params['phi'], brain_params['theta'], brain_params['omega'], brain_params['alpha'])
print(f'rcos_phi shape: {rcos_phi_final.shape}')

In [ ]:
# Stage 3: Differentiable feedback (NO reset_weights)
feedback_mlp = FeedbackMLP().to(device)
heart_osc_torch = HeartOscillatorTorch(omega1_hz=dyn['omega1_hz'], omega2_hz=dyn['omega2_hz']).to(device)
opt = optim.Adam(list(heart_model.parameters()) + list(feedback_mlp.parameters()), lr=tr['feedback_lr'])
ecg_tgt = torch.tensor(ecg_processed[::10], dtype=torch.float32, device=device).unsqueeze(1)
rcos_phi_dev = rcos_phi_final.detach().to(device)
feedback_losses = []
for ep in range(tr['feedback_epochs']):
    modulation = feedback_mlp(rcos_phi_dev.unsqueeze(-1))
    heart_traj = heart_osc_torch(T=t_duration, dt=1/fs, modulation=modulation)
    loss = nn.MSELoss()(heart_model(heart_traj), ecg_tgt)
    opt.zero_grad()
    loss.backward()
    opt.step()
    feedback_losses.append(loss.item())
    if (ep+1) % 500 == 0:
        print(f'Feedback Epoch {ep+1}, Loss: {loss.item():.6f}')
print('Feedback done.')

In [ ]:
# Final predictions and plotting
heart_model.eval()
feedback_mlp.eval()
with torch.no_grad():
    sim_baseline = simulate_coupled_oscillators_numpy(T=t_duration, dt=1/fs, omega1_hz=dyn['omega1_hz'], omega2_hz=dyn['omega2_hz'])
    pred_ecg_baseline = heart_model(torch.tensor(sim_baseline, dtype=torch.float32, device=device)).cpu().numpy().flatten()
    modulation = feedback_mlp(rcos_phi_final.detach().to(device).unsqueeze(-1))
    heart_traj = heart_osc_torch(T=t_duration, dt=1/fs, modulation=modulation)
    pred_ecg_feedback = heart_model(heart_traj).cpu().numpy().flatten()
    P_out_baseline = torch.sum(alpha_final * r_final * torch.cos(phi_final), dim=1).cpu().numpy()

target_ecg = ecg_processed[::10]
D_func = interp1d(t, target_signal, kind='linear', bounds_error=False, fill_value=0)

fig, axes = plt.subplots(5, 1, figsize=(15, 20))
axes[0].plot(brain_losses); axes[0].set_title('Stage 1: Brain Loss'); axes[0].grid(True)
axes[1].plot(mlp_losses); axes[1].set_title('Stage 2: MLP Loss'); axes[1].grid(True)
axes[2].plot(feedback_losses); axes[2].set_title('Stage 3: Feedback Loss'); axes[2].grid(True)
ts = np.linspace(0, t_duration, len(target_ecg))
axes[3].plot(ts, target_ecg, label='Target ECG', lw=2)
axes[3].plot(ts, pred_ecg_baseline, '--', label='Baseline ECG')
axes[3].plot(ts, pred_ecg_feedback, ':', label='Feedback ECG')
axes[3].set_title('ECG Prediction'); axes[3].legend(); axes[3].grid(True)
axes[4].plot(t, D_func(t), label='Target EEG', lw=2)
axes[4].plot(t, P_out_baseline, label='P_out baseline', alpha=0.7)
axes[4].set_title('Brain Output'); axes[4].legend(); axes[4].grid(True)
plt.tight_layout()
Path('figures').mkdir(exist_ok=True)
plt.savefig('figures/full_feedback_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Done. Check figures/')